# 009 — Export the model

Step 1.8. Train the final XGBoost model, measure it on the test set, and save it with metadata so the API can load it. **Done when** a brand-new Python process, with none of this notebook's state, loads the files and scores a transaction to the same probability.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import json
import subprocess
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import settings
from src.ml.artifact import save_model
from src.ml.evaluate import bootstrap_pr_auc, summary_metrics
from src.ml.model import VALIDATION_SIZE, XGB_PARAMS, fit_xgboost, fraud_probability
from src.ml.preprocess import load_splits
from src.ml.thresholds import approve_all_cost_per_100k, costs_from_settings, policy_costs

## 2. Train the final model

In [2]:
X_train, X_test, y_train, y_test = load_splits()

start = time.perf_counter()
model = fit_xgboost(X_train, y_train)
xgb = model.named_steps["model"]
print(f"trained in {time.perf_counter() - start:.1f}s; early stopping kept {xgb.best_iteration + 1} trees")

test_proba = fraud_probability(model, X_test)

trained in 4.5s; early stopping kept 120 trees


## 3. Test-set metrics to record

In [3]:
metrics = summary_metrics(y_test, test_proba)
ci_low, ci_high = bootstrap_pr_auc(y_test, test_proba, seed=settings.random_seed)

costs = costs_from_settings()
policy = policy_costs(
    y_test, test_proba, X_test["Amount"], costs,
    review_candidates=[settings.review_threshold],
    block_candidates=[settings.block_threshold],
).iloc[0]
no_model = approve_all_cost_per_100k(y_test, X_test["Amount"], costs)

test_metrics = {
    "rows": metrics["n"],
    "fraud": metrics["n_fraud"],
    "pr_auc": metrics["pr_auc"],
    "pr_auc_95ci": [ci_low, ci_high],
    "pr_auc_random_baseline": metrics["pr_auc_baseline"],
    "roc_auc": metrics["roc_auc"],
    "at_decision_thresholds": {
        "reviewed_rate": policy["review_rate"],
        "blocked_rate": policy["block_rate"],
        "fraud_stopped": policy["fraud_stopped"],
        "fraud_loss_stopped": policy["fraud_loss_stopped"],
        "cost_per_100k": policy["cost_per_100k"],
        "saving_vs_no_model": 1 - policy["cost_per_100k"] / no_model,
    },
}
print(json.dumps(test_metrics, indent=2))

{
  "rows": 56746,
  "fraud": 95,
  "pr_auc": 0.820699719493611,
  "pr_auc_95ci": [
    0.7447611244254364,
    0.8907100075455288
  ],
  "pr_auc_random_baseline": 0.0016741268107003137,
  "roc_auc": 0.969557094267858,
  "at_decision_thresholds": {
    "reviewed_rate": 0.0008282522116096288,
    "blocked_rate": 0.0005639163993937899,
    "fraud_stopped": 0.7578947368421053,
    "fraud_loss_stopped": 0.7432820445041197,
    "cost_per_100k": 7739.047686180528,
    "saving_vs_no_model": 0.728768086090625
  }
}


## 4. Save the model and its metadata

In [4]:
metadata = {
    "model_type": "xgboost",
    "risk_score": "predicted probability that the transaction is fraud",
    "decision_thresholds": {
        "review": settings.review_threshold,
        "block": settings.block_threshold,
        "chosen_in": "notebooks/008_xgboost_thresholds.ipynb",
    },
    "cost_assumptions": {
        "review_cost": costs.review_cost,
        "false_block_cost": costs.false_block_cost,
        "chargeback_fee": costs.chargeback_fee,
        "max_review_rate": settings.max_review_rate,
    },
    "training": {
        "data": settings.raw_data_filename,
        "deduplicated": True,
        "train_rows": len(X_train),
        "train_fraud": int(y_train.sum()),
        "test_size": settings.test_size,
        "random_seed": settings.random_seed,
        "early_stopping_validation_size": VALIDATION_SIZE,
        "trees": int(xgb.best_iteration + 1),
        "xgboost_params": XGB_PARAMS,
    },
    "test_metrics": test_metrics,
}

saved = save_model(model, metadata)
print(f"model:    {settings.model_path}  ({settings.model_path.stat().st_size / 1e3:.0f} KB)")
print(f"metadata: {settings.model_metadata_path}")
print(f"version:  {saved['model_version']}")

model:    D:\Projects\fraud-detection-api\models\fraud_model.joblib  (324 KB)
metadata: D:\Projects\fraud-detection-api\models\model_metadata.json
version:  20260915-070953-19429cbb


In [5]:
print(settings.model_metadata_path.read_text(encoding="utf-8"))

{
  "model_version": "20260915-070953-19429cbb",
  "created_at": "2026-09-15T07:09:53+00:00",
  "model_file": "fraud_model.joblib",
  "model_sha256": "19429cbbf5bffabb8c1d8cf4a085671cab6f0d48683173ff9a4062452713effb",
  "input_features": [
    "Time",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
    "V7",
    "V8",
    "V9",
    "V10",
    "V11",
    "V12",
    "V13",
    "V14",
    "V15",
    "V16",
    "V17",
    "V18",
    "V19",
    "V20",
    "V21",
    "V22",
    "V23",
    "V24",
    "V25",
    "V26",
    "V27",
    "V28",
    "Amount"
  ],
  "model_features": [
    "Amount",
    "time_sin",
    "time_cos",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
    "V7",
    "V8",
    "V9",
    "V10",
    "V11",
    "V12",
    "V13",
    "V14",
    "V15",
    "V16",
    "V17",
    "V18",
    "V19",
    "V20",
    "V21",
    "V22",
    "V23",
    "V24",
    "V25",
    "V26",
    "V27",
    "V28"
  ],
  "library_versions": {
    "python": "3.10.9",
    "scikit-

## 5. Load and score in a brand-new Python process

The script below runs in a separate interpreter. It knows nothing about this notebook: it only has the two files on disk and the project code.

In [6]:
FRESH_PROCESS = """
import json
import sys

import pandas as pd

from src.ml.artifact import load_model

pipeline, metadata = load_model()
row = pd.DataFrame([json.loads(sys.argv[1])], columns=metadata["input_features"])
probability = float(pipeline.predict_proba(row)[0, 1])
thresholds = metadata["decision_thresholds"]
if thresholds["block"] is not None and probability >= thresholds["block"]:
    decision = "block"
elif probability >= thresholds["review"]:
    decision = "review"
else:
    decision = "approve"
print(json.dumps({"model_version": metadata["model_version"], "fraud_probability": probability, "decision": decision}))
"""

block_at = np.inf if settings.block_threshold is None else settings.block_threshold
review_band = np.flatnonzero((test_proba >= settings.review_threshold) & (test_proba < block_at))
examples = {
    "highest-risk test transaction": X_test.iloc[[int(np.argmax(test_proba))]],
    "a transaction in the review band": X_test.iloc[[int(review_band[0])]],
    "an ordinary transaction": X_test.iloc[[0]],
}

for label, row in examples.items():
    result = subprocess.run(
        [sys.executable, "-c", FRESH_PROCESS, json.dumps(row.iloc[0].to_dict())],
        cwd=PROJECT_ROOT, capture_output=True, text=True, check=True,
    )
    scored = json.loads(result.stdout)
    in_notebook = float(fraud_probability(model, row)[0])
    assert abs(scored["fraud_probability"] - in_notebook) < 1e-6
    print(f"{label}: {scored['decision']:7s} p={scored['fraud_probability']:.6f} "
          f"(notebook: {in_notebook:.6f}) version {scored['model_version']}")

highest-risk test transaction: block   p=0.986627 (notebook: 0.986627) version 20260915-070953-19429cbb


a transaction in the review band: review  p=0.845157 (notebook: 0.845157) version 20260915-070953-19429cbb


an ordinary transaction: approve p=0.000064 (notebook: 0.000064) version 20260915-070953-19429cbb


## Step 1.8 findings

**Saved**
- `models/fraud_model.joblib` (324 KB): the fitted pipeline, preprocessing plus XGBoost with 120 trees.
- `models/model_metadata.json` (2.6 KB): version `20260915-070953-19429cbb`.
- Both files are gitignored. They are rebuilt by training, not committed.

**What the metadata records**
- The model file's SHA-256, the order of the 30 raw input columns, and the 31 features the model actually sees.
- Library versions (Python 3.10.9, scikit-learn 1.7.2, XGBoost 3.2.0, pandas 2.3.3, NumPy 2.2.6).
- Decision thresholds (review 0.24, block 0.95) and the notebook that chose them, plus the cost assumptions behind them.
- Training details (226,980 rows, 378 fraud, seed 42) and test metrics.

**Reproducible:** the test metrics are identical to notebook 008 (PR-AUC 0.821, 95% CI 0.745 to 0.891; 73% saving vs no model), so retraining gives the same model.

**Done-when met:** a separate Python process loaded the two files and scored three test transactions to the same probabilities as this notebook, and applied the thresholds:

| transaction | probability | decision |
|---|---|---|
| highest-risk in the test set | 0.9866 | block |
| one in the review band | 0.8452 | review |
| an ordinary one | 0.000064 | approve |

**Safety checks on load**
- The SHA-256 is checked *before* unpickling, so a mismatched or corrupted model file is refused without running it.
- A different input feature order is refused.
- Different library versions raise a warning.
- Unpickling runs code from the file, so only load model files you created.

**For Phase 2:** thresholds now exist in two places: `.env` (what the API will apply) and the metadata (what was chosen for this model). The API should warn at startup if they differ.